# HARIBON Objective 2 – XGBoost Red Tide Risk Prediction

**Goal:** Train a binary-classification XGBoost to predict next-day red-tide risk from multi-source oceanographic/atmospheric features.

**Pipeline steps:**
1. Data loading & exploration
2. Hybrid Gap-Adaptive Imputation (micro / block / seasonal)
3. Label binarisation & feature selection
4. Rolling-Origin Cross-Validation with temporal splits
5. XGBoost with hyperparameter optimization
6. Per-split & aggregate metric reporting

**Configuration:** 6 temporal splits, metrics: Accuracy, Precision, Recall, F1-Score, AUC-ROC

In [1]:
# ──────────────────────────────────────────────
# Cell 1 – Imports & Global Configuration
# ──────────────────────────────────────────────
import os, warnings, random, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from datetime import datetime

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ──
DATA_PATH = os.path.join("..", "final_compiled_dataset", "Combined_Labeled.csv")
MODEL_DIR = os.path.join("saved_model")
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Hyper-parameters ──
# Hyperparameter search grid
PARAM_GRID = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'n_estimators': [50, 100, 200, 300, 500],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 5, 10, 20, 50, 100],  # Handle class imbalance
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4]
}

# ── Feature list (drop NDVI_raw — 97 % NaN, drop red_tide source column) ──
FEATURES = [
    "CHL", "NDVI_daily", "mlotst", "precip_mm_day",
    "so", "thetao", "uo", "vo",
    "wind_speed_ms", "wind_u_ms", "wind_v_ms",
]
TARGET = "red_tide_label"

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Target: {TARGET}")

Features (11): ['CHL', 'NDVI_daily', 'mlotst', 'precip_mm_day', 'so', 'thetao', 'uo', 'vo', 'wind_speed_ms', 'wind_u_ms', 'wind_v_ms']
Target: red_tide_label


## 1 · Data Loading & Initial Exploration

In [2]:
# ──────────────────────────────────────────────
# Cell 2 – Load & quick-look
# ──────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df_raw.sort_values(["Location_Name", "Date"], inplace=True)
df_raw.reset_index(drop=True, inplace=True)

print("Shape:", df_raw.shape)
print("Locations:", df_raw["Location_Name"].nunique())
print("\n── Label distribution (raw) ──")
print(df_raw[TARGET].value_counts(dropna=False))
print("\n── NaN per column ──")
print(df_raw.isnull().sum())
print("\n── Date range ──")
print(df_raw["Date"].min(), "→", df_raw["Date"].max())

Shape: (28511, 16)
Locations: 7

── Label distribution (raw) ──
red_tide_label
0.000    23031
1.000     4801
NaN        495
0.800       30
0.500       28
0.650       24
0.550       21
0.600       21
0.700       21
0.750       21
0.560        3
0.620        3
0.680        3
0.740        3
0.575        3
0.725        3
Name: count, dtype: int64

── NaN per column ──
Location_Name         0
red_tide          26089
Date                  0
CHL                 168
NDVI_daily        10135
NDVI_raw          27705
mlotst              441
precip_mm_day       245
so                  441
thetao              441
uo                  441
vo                  441
wind_speed_ms        77
wind_u_ms            77
wind_v_ms            77
red_tide_label      495
dtype: int64

── Date range ──
2015-01-01 00:00:00 → 2026-02-24 00:00:00


## 2 · Hybrid Gap-Adaptive Imputation

| Gap length | Strategy |
|---|---|
| **< 7 days** (micro) | Linear interpolation (time-based) |
| **7 – 30 days** (block) | Polynomial interpolation (order 2) |
| **> 30 days** (seasonal / long) | Climatological mean (month-day average for that location across all years) |

Imputation is applied **per location, per feature column** and avoids future-data leakage by only using data up to the gap's year.

In [3]:
# ──────────────────────────────────────────────
# Cell 3 – Hybrid Gap-Adaptive Imputation
# ──────────────────────────────────────────────
from sklearn.impute import KNNImputer
from scipy import interpolate

def _identify_nan_gaps(series: pd.Series) -> list[dict]:
    """Return a list of dicts describing contiguous NaN blocks.
    Each dict: {'start': int, 'end': int, 'length': int}  (iloc positions).
    """
    is_nan = series.isna().values
    gaps = []
    i = 0
    while i < len(is_nan):
        if is_nan[i]:
            start = i
            while i < len(is_nan) and is_nan[i]:
                i += 1
            gaps.append({"start": start, "end": i - 1, "length": i - start})
        else:
            i += 1
    return gaps


def _climatological_mean(loc_df: pd.DataFrame, col: str) -> pd.Series:
    """Month-Day average across all years for *col* within one location."""
    loc_df = loc_df.copy()
    loc_df["_md"] = loc_df["Date"].dt.strftime("%m-%d")
    clim = loc_df.groupby("_md")[col].transform("mean")
    return clim


def hybrid_gap_adaptive_impute(df, features, target_col):
    """Apply hybrid gap-adaptive imputation per location.
    
    Strategy per gap length:
      micro  (< 7 days)  → linear interpolation (time-based)
      block  (7-30 days) → polynomial interpolation (order=2)
      long   (> 30 days) → climatological mean (month-day avg per location)
    """
    df = df.copy()
    df = df.sort_values(["Location_Name", "Date"]).reset_index(drop=True)

    for location in df["Location_Name"].unique():
        loc_mask = df["Location_Name"] == location
        idx = df[loc_mask].index
        
        for col in features:
            if col == target_col:
                continue
            
            ts = pd.to_numeric(df.loc[idx, col], errors="coerce")
            if ts.isna().sum() == 0:
                continue

            gaps = _identify_nan_gaps(ts)
            if not gaps:
                continue

            # Pre-compute climatological fill for this column / location
            clim_fill = _climatological_mean(df.loc[idx], col)

            for gap in gaps:
                g_len = gap["length"]
                gap_iloc_start = gap["start"]
                gap_iloc_end = gap["end"] + 1

                if g_len < 7:
                    # Micro gap → linear interpolation (time-based)
                    ts = ts.interpolate(method="linear", limit_direction="both")
                elif g_len <= 30:
                    # Block gap → polynomial order-2
                    ts = ts.interpolate(method="polynomial", order=2, limit_direction="both")
                else:
                    # Long / seasonal gap → climatological mean
                    ts.iloc[gap_iloc_start:gap_iloc_end] = clim_fill.iloc[gap_iloc_start:gap_iloc_end]

            # Final safety net: any remaining NaN filled with forward-fill then back-fill
            ts = ts.ffill().bfill()
            
            df.loc[idx, col] = ts
    
    return df

# ── Run imputation ──
print("NaN BEFORE imputation:")
print(df_raw[FEATURES].isnull().sum())

print("\nApplying hybrid gap-adaptive imputation...")
df_imputed = hybrid_gap_adaptive_impute(df_raw, FEATURES + [TARGET], TARGET)

print("\nNaN AFTER imputation:")
print(df_imputed[FEATURES].isnull().sum())

NaN BEFORE imputation:
CHL                168
NDVI_daily       10135
mlotst             441
precip_mm_day      245
so                 441
thetao             441
uo                 441
vo                 441
wind_speed_ms       77
wind_u_ms           77
wind_v_ms           77
dtype: int64

Applying hybrid gap-adaptive imputation...

NaN AFTER imputation:
CHL              0
NDVI_daily       0
mlotst           0
precip_mm_day    0
so               0
thetao           0
uo               0
vo               0
wind_speed_ms    0
wind_u_ms        0
wind_v_ms        0
dtype: int64


## 3 · Label Binarisation & Feature Selection

In [4]:
# ──────────────────────────────────────────────
# Cell 4 – Label binarisation
# ──────────────────────────────────────────────
df = df_imputed.dropna(subset=[TARGET]).copy()
df['target'] = (df[TARGET] >= 0.5).astype(int)

print("Label distribution after binarisation:")
print(df['target'].value_counts())

# Feature selection
X = df[FEATURES]
y = df['target']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

Label distribution after binarisation:
target
0    23031
1     4985
Name: count, dtype: int64

Features shape: (28016, 11)
Target shape: (28016,)


## 4 · Rolling-Origin Cross-Validation with Temporal Splits

In [5]:
# ──────────────────────────────────────────────
# Cell 5 – Define temporal splits
# ──────────────────────────────────────────────
# Define temporal splits: (train_end_year_inclusive, test_year(s))
SPLITS = [
    (2019, [2020]),
    (2020, [2021]),
    (2021, [2022]),
    (2022, [2023]),
    (2023, [2024]),
    (2024, [2025, 2026]),
]

print("Temporal splits defined:")
for i, (train_end, test_years) in enumerate(SPLITS, 1):
    print(f"Split {i}: Train ≤ {train_end}, Test = {test_years}")

Temporal splits defined:
Split 1: Train ≤ 2019, Test = [2020]
Split 2: Train ≤ 2020, Test = [2021]
Split 3: Train ≤ 2021, Test = [2022]
Split 4: Train ≤ 2022, Test = [2023]
Split 5: Train ≤ 2023, Test = [2024]
Split 6: Train ≤ 2024, Test = [2025, 2026]


## 5 · XGBoost Training & Evaluation

In [6]:
# ──────────────────────────────────────────────
# Cell 6 – Train and evaluate per split
# ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

results = []

for split_idx, (train_end_yr, test_yrs) in enumerate(SPLITS, 1):
    print(f"\n--- Split {split_idx}: Train ≤ {train_end_yr}, Test = {test_yrs} ---")
    
    # Split data temporally
    train_mask = df['Date'].dt.year <= train_end_yr
    test_mask = df['Date'].dt.year.isin(test_yrs)
    
    X_train = X[train_mask]
    y_train = y[train_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]
    
    print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
    print(f"Train positive rate: {y_train.mean():.3f}, Test positive rate: {y_test.mean():.3f}")
    
    # Hyperparameter optimization
    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=SEED
    )
    
    random_search = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=PARAM_GRID,
        n_iter=20,  # Reduced for speed
        scoring='roc_auc',
        cv=3,
        random_state=SEED,
        n_jobs=-1
    )
    
    print("Optimizing hyperparameters...")
    random_search.fit(X_train, y_train)
    
    # Best model
    best_model = random_search.best_estimator_
    
    # Evaluate
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_pred_proba) if len(y_test.unique()) > 1 else np.nan
    }
    
    print(f"AUC: {metrics['auc']:.4f}, F1: {metrics['f1']:.4f}")
    
    results.append({
        'split': split_idx,
        'train_end': train_end_yr,
        'test_years': test_yrs,
        **metrics,
        'best_params': random_search.best_params_
    })

print("\n✅ All splits complete.")


--- Split 1: Train ≤ 2019, Test = [2020] ---
Train samples: 12644, Test samples: 2484
Train positive rate: 0.097, Test positive rate: 0.057
Optimizing hyperparameters...
AUC: 0.6544, F1: 0.1217

--- Split 2: Train ≤ 2020, Test = [2021] ---
Train samples: 15128, Test samples: 2505
Train positive rate: 0.090, Test positive rate: 0.196
Optimizing hyperparameters...
AUC: 0.6042, F1: 0.3160

--- Split 3: Train ≤ 2021, Test = [2022] ---
Train samples: 17633, Test samples: 2476
Train positive rate: 0.105, Test positive rate: 0.317
Optimizing hyperparameters...
AUC: 0.6652, F1: 0.4856

--- Split 4: Train ≤ 2022, Test = [2023] ---
Train samples: 20109, Test samples: 2504
Train positive rate: 0.131, Test positive rate: 0.412
Optimizing hyperparameters...
AUC: 0.6231, F1: 0.5739

--- Split 5: Train ≤ 2023, Test = [2024] ---
Train samples: 22613, Test samples: 2520
Train positive rate: 0.162, Test positive rate: 0.235
Optimizing hyperparameters...
AUC: 0.7654, F1: 0.4149

--- Split 6: Train ≤ 202

## 7 · Aggregate Metrics

In [7]:
# ──────────────────────────────────────────────
# Cell 7 – Aggregate results
# ──────────────────────────────────────────────
results_df = pd.DataFrame(results)
print("Per-split metrics:")
print(results_df)

print("\nAggregate metrics (mean ± std):")
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    mean_val = results_df[col].mean()
    std_val = results_df[col].std()
    print(f"{col}: {mean_val:.4f} ± {std_val:.4f}")

# Save results
results_df.to_csv(os.path.join("results", "xgboost_split_metrics.csv"), index=False)
print("\nResults saved to results/xgboost_split_metrics.csv")

Per-split metrics:
   split  train_end    test_years  accuracy  precision    recall        f1  \
0      1       2019        [2020]  0.587359   0.069268  0.500000  0.121680   
1      2       2020        [2021]  0.605988   0.239748  0.463415  0.316008   
2      3       2021        [2022]  0.539580   0.375698  0.686224  0.485560   
3      4       2022        [2023]  0.501198   0.442632  0.815713  0.573866   
4      5       2023        [2024]  0.412302   0.270898  0.885329  0.414856   
5      6       2024  [2025, 2026]  0.300035   0.263235  1.000000  0.416763   

        auc                                        best_params  
0  0.654387  {'subsample': 0.9, 'scale_pos_weight': 20, 'n_...  
1  0.604226  {'subsample': 0.8, 'scale_pos_weight': 20, 'n_...  
2  0.665218  {'subsample': 0.6, 'scale_pos_weight': 100, 'n...  
3  0.623084  {'subsample': 0.9, 'scale_pos_weight': 20, 'n_...  
4  0.765432  {'subsample': 0.8, 'scale_pos_weight': 100, 'n...  
5  0.933036  {'subsample': 0.6, 'scale_pos_w

In [8]:
# ──────────────────────────────────────────────
# Cell 8 – Retrain final model on full data & save artefact
# ──────────────────────────────────────────────

# Rebuild results table so we can reuse the best split's tuned parameters.
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No split results available; run the cross-validation cell first.")

best_result_row = results_df.loc[results_df['auc'].idxmax()]
best_params = dict(best_result_row['best_params'])

# Ensure the model stays aligned with the binary objective used during evaluation.
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': SEED,
})

print("Retraining final XGBoost model on the full dataset...")
print(f"Best split selected for parameters: #{int(best_result_row['split'])} with AUC {best_result_row['auc']:.4f}")
print(f"Final parameters: {best_params}")

final_model = XGBClassifier(**best_params)
final_model.fit(X, y)

model_path = os.path.join(MODEL_DIR, "best_xgboost_model.json")
final_model.save_model(model_path)

print(f"\n✅ Final model saved to: {model_path}")


Retraining final XGBoost model on the full dataset...
Best split selected for parameters: #6 with AUC 0.9330
Final parameters: {'subsample': 0.6, 'scale_pos_weight': 50, 'n_estimators': 50, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.4, 'colsample_bytree': 0.6, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'random_state': 42}

✅ Final model saved to: saved_model\best_xgboost_model.json


## 8 · XGBoost Model Summary

In [9]:
# ──────────────────────────────────────────────
# Cell 9 – XGBoost Model Summary
# ──────────────────────────────────────────────

print("XGBOOST MODEL FINAL SUMMARY")
print("=" * 50)

# Base results table
results_df = pd.DataFrame(results)

# Aggregate performance table
agg_metrics = []
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    agg_metrics.append({
        'Metric': col.capitalize(),
        'Mean': round(results_df[col].mean(), 4),
        'Std': round(results_df[col].std(), 4),
    })
perf_df = pd.DataFrame(agg_metrics)

print("\nAggregate Performance:")
display(perf_df)

# Split comparison table (new)
split_compare_df = results_df[
    ['split', 'train_end', 'test_years', 'accuracy', 'precision', 'recall', 'f1', 'auc']
].copy()
split_compare_df = split_compare_df.sort_values('split').reset_index(drop=True)
split_compare_df = split_compare_df.round(4)

print("\nSplit Comparison:")
display(split_compare_df)

best_split = split_compare_df.loc[split_compare_df['auc'].idxmax()]
print(
    f"\nBest Split by AUC: #{int(best_split['split'])} "
    f"(Train ≤ {int(best_split['train_end'])}, Test = {best_split['test_years']})"
)
print(f"AUC-ROC: {best_split['auc']:.4f}")

if 'model_path' in globals():
    print(f"\nSaved final model: {model_path}")

print("\nResults saved to: results/xgboost_split_metrics.csv")

XGBOOST MODEL FINAL SUMMARY

Aggregate Performance:


,Metric,Mean,Std
0,Accuracy,0.4911,0.1163
1,Precision,0.2769,0.1280
2,Recall,0.7251,0.2145
3,F1,0.3881,0.1560
4,Auc,0.7076,0.1238



Split Comparison:


,split,train_end,test_years,accuracy,precision,recall,f1,auc
0,1,2019,[2020],0.5874,0.0693,0.5000,0.1217,0.6544
1,2,2020,[2021],0.6060,0.2397,0.4634,0.3160,0.6042
2,3,2021,[2022],0.5396,0.3757,0.6862,0.4856,0.6652
3,4,2022,[2023],0.5012,0.4426,0.8157,0.5739,0.6231
4,5,2023,[2024],0.4123,0.2709,0.8853,0.4149,0.7654
5,6,2024,"[2025, 2026]",0.3000,0.2632,1.0000,0.4168,0.9330



Best Split by AUC: #6 (Train ≤ 2024, Test = [2025, 2026])
AUC-ROC: 0.9330

Saved final model: saved_model\best_xgboost_model.json

Results saved to: results/xgboost_split_metrics.csv
